# Lab 05 – Image Segmentation (Python · OpenCV · NumPy)

This notebook is built up **one task per step**. Each task is appended below the previous one.

**General requirements followed in every task**
- Only Python, OpenCV, NumPy (+ Matplotlib for display) are used – no prebuilt segmentation library or deep-learning model.
- The original image and the segmentation result are always displayed.
- Selected parameters are recorded and briefly justified.
- The final output of each task is saved to the `outputs/` folder.

| Task | Topic | Status |
|---|---|---|
| 1 | Document under uneven lighting – global vs adaptive thresholding | ✅ done |
| 2 – 10 | *(appended in later steps)* | ⏳ |

---
# Task 1 — Automated Inspection of a Document Under Uneven Lighting

**Goal:** separate printed text from background when one side of the page is brighter than the other.

**Image:** `sudoku.png` (OpenCV sample image – place it in the same folder as this notebook, or change `IMAGE_PATH`).

**Plan**
1. Load in grayscale  →  2. Global thresholding with ≥ 3 thresholds  →  3. Compare masks  →
4. Adaptive thresholding  →  5. Adaptive vs best global  →  6. Find regions where global thresholding fails  →
7. Select the most usable parameter combination  →  required comparison figure + answer.

In [ ]:
import os, json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import textwrap

IMAGE_PATH = "sudoku.png"      # <- change if your file lives elsewhere
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["image.cmap"] = "gray"

def show(ax, img, title, cmap="gray"):
    ax.imshow(img, cmap=cmap, vmin=0 if cmap == "gray" else None, vmax=255 if cmap == "gray" else None)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])          # keep a visible frame so white masks do not blend into the page
    for sp in ax.spines.values():
        sp.set_edgecolor("0.4"); sp.set_linewidth(1)

print("OpenCV :", cv2.__version__)
print("NumPy  :", np.__version__)

## Step 1 — Load the image in grayscale

In [ ]:
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
if gray is None:
    raise FileNotFoundError(
        f"Could not read '{IMAGE_PATH}'. Download sudoku.png (OpenCV samples/data/sudoku.png) "
        "and put it next to this notebook, or update IMAGE_PATH."
    )
H, W = gray.shape
print(f"Image size: {W} x {H}  | dtype: {gray.dtype}  | min/max: {gray.min()}/{gray.max()}")

# Look at the lighting: histogram + average brightness of every column / row.
col_profile = gray.mean(axis=0)
row_profile = gray.mean(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
show(axes[0], gray, "Original (grayscale)")
axes[1].hist(gray.ravel(), bins=64, range=(0, 255), color="gray")
axes[1].set_title("Grayscale histogram")
axes[1].set_xlabel("intensity"); axes[1].set_ylabel("pixels")
axes[2].plot(col_profile, label="mean per column (left→right)")
axes[2].plot(row_profile, label="mean per row (top→bottom)")
axes[2].set_title("Average brightness profile")
axes[2].set_xlabel("position (px)"); axes[2].set_ylabel("mean intensity")
axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"Column-profile range : {col_profile.min():.0f} … {col_profile.max():.0f}  (spread {col_profile.max()-col_profile.min():.0f})")
print(f"Row-profile range    : {row_profile.min():.0f} … {row_profile.max():.0f}  (spread {row_profile.max()-row_profile.min():.0f})")

A large spread in the brightness profile means the *paper itself* changes brightness across the page – so no single
intensity value can separate "ink" from "paper" everywhere.

## Step 2 — Global thresholding with three different threshold values

The text is **dark on a light page**, so I use `THRESH_BINARY_INV`: pixels **darker** than `T` become white (255 = "text"), everything else becomes black.

In [ ]:
# Three global thresholds: low / medium / high  (tune after looking at the histogram above)
GLOBAL_T = {"T1": 90, "T2": 127, "T3": 160}

global_masks = {}
for name, t in GLOBAL_T.items():
    _, m = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY_INV)
    global_masks[name] = m

fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
show(axes[0], gray, "Original")
for ax, (name, m) in zip(axes[1:], global_masks.items()):
    show(ax, m, f"Global {name}  (T = {GLOBAL_T[name]})")
plt.tight_layout(); plt.show()

## Step 3 — Compare the global binary masks

Without a ground-truth, I build a **reference mask** only to *measure* the comparison.
It is made by estimating the slowly-varying page brightness (`background`), dividing it out (illumination normalisation) and then applying Otsu.
It is a **proxy**, not true ground truth – it is used for ranking/diagnosis, and every conclusion is also checked visually.

In [ ]:
# --- illumination (background) estimate: remove thin dark text, then smooth heavily
k = max(3, (min(H, W) // 25) | 1)                         # odd kernel ~4 % of the image side
bg = cv2.dilate(gray, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))   # erase thin dark strokes
bg = cv2.medianBlur(bg, k if k % 2 == 1 else k + 1)       # smooth -> page brightness map

# --- illumination-normalised image and proxy reference mask (text = 255)
norm = cv2.divide(gray, bg, scale=255)
_, ref_mask = cv2.threshold(norm, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
show(axes[0], bg, "Estimated page illumination (background)")
show(axes[1], norm, "Illumination-normalised image")
show(axes[2], ref_mask, "Proxy reference mask (for measuring only)")
plt.tight_layout(); plt.show()

def scores(pred, ref):
    p, r = pred > 0, ref > 0
    tp = np.sum(p & r); fp = np.sum(p & ~r); fn = np.sum(~p & r)
    prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
    return {"precision": prec, "recall": rec,
            "f1": 2 * prec * rec / (prec + rec + 1e-9),
            "iou": tp / (tp + fp + fn + 1e-9),
            "text_%": 100 * p.mean()}

print(f"{'mask':<12}{'text %':>8}{'precision':>11}{'recall':>9}{'F1':>8}{'IoU':>8}")
print("-" * 56)
print(f"{'reference':<12}{100*(ref_mask>0).mean():>8.1f}")
global_scores = {}
for name, m in global_masks.items():
    s = scores(m, ref_mask); global_scores[name] = s
    print(f"{name+' (T='+str(GLOBAL_T[name])+')':<12}{s['text_%']:>8.1f}{s['precision']:>11.3f}{s['recall']:>9.3f}{s['f1']:>8.3f}{s['iou']:>8.3f}")

best_global_name = max(global_scores, key=lambda n: global_scores[n]["f1"])
best_global_T = GLOBAL_T[best_global_name]
best_global_mask = global_masks[best_global_name]
print(f"\nBest global threshold (highest F1): {best_global_name}  (T = {best_global_T})")

**How to read the table**
- Low `T` → very few pixels become "text": high precision but **low recall** (strokes in brighter areas survive, strokes in dim areas are lost).
- High `T` → many pixels become "text": high recall but **low precision** (dim paper is swallowed as black blobs).
- No single `T` gets both – that is the failure of a global threshold under uneven light.

## Step 4 — Adaptive thresholding on the same image

Adaptive thresholding computes a **separate threshold for each pixel** from its local neighbourhood:
`T(x, y) = local_weighted_mean(x, y) − C`. Initial values below follow the common OpenCV starting point (blockSize = 11, C = 2); the best combination is chosen in Step 7.

In [ ]:
ADAPT_BLOCK, ADAPT_C = 11, 2          # initial (default-style) setting

adaptive_initial = cv2.adaptiveThreshold(
    gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,
    blockSize=ADAPT_BLOCK, C=ADAPT_C)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
show(axes[0], gray, "Original")
show(axes[1], best_global_mask, f"Best global ({best_global_name}, T={best_global_T})")
show(axes[2], adaptive_initial, f"Adaptive Gaussian (block={ADAPT_BLOCK}, C={ADAPT_C})")
plt.tight_layout(); plt.show()

## Step 5 — Adaptive result vs. best global result

In [ ]:
s_g = scores(best_global_mask, ref_mask)
s_a = scores(adaptive_initial, ref_mask)
print(f"{'method':<34}{'text %':>8}{'precision':>11}{'recall':>9}{'F1':>8}")
print("-" * 70)
for label, s in [(f"Best global (T={best_global_T})", s_g), (f"Adaptive Gaussian ({ADAPT_BLOCK}, C={ADAPT_C})", s_a)]:
    print(f"{label:<34}{s['text_%']:>8.1f}{s['precision']:>11.3f}{s['recall']:>9.3f}{s['f1']:>8.3f}")

# Side-by-side difference to the reference: red = unwanted text-pixels (false +), blue = missing text-pixels (false −)
def error_overlay(pred, ref):
    p, r = pred > 0, ref > 0
    out = np.full((*ref.shape, 3), 255, np.uint8)
    out[p & r]  = (0, 0, 0)          # correct text -> black
    out[p & ~r] = (255, 0, 0)        # unwanted     -> red
    out[~p & r] = (0, 90, 255)       # missing      -> blue
    return out

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(error_overlay(best_global_mask, ref_mask)); axes[0].set_title(f"Best global vs reference\n(black=ok, red=unwanted, blue=missing)"); axes[0].axis("off")
axes[1].imshow(error_overlay(adaptive_initial, ref_mask)); axes[1].set_title(f"Adaptive vs reference\n(black=ok, red=unwanted, blue=missing)"); axes[1].axis("off")
plt.tight_layout(); plt.show()

## Step 6 — Which regions of the document make global thresholding fail?

The page is split into a **3 × 3 grid**. For each cell I measure (a) the local paper brightness and
(b) how many pixels the best global mask got wrong: **unwanted** (background marked as text) or **missing** (text lost).

In [ ]:
rows_n = ["top", "middle", "bottom"]; cols_n = ["left", "centre", "right"]
N = 3
p, r = best_global_mask > 0, ref_mask > 0
fp_map = np.zeros((N, N)); fn_map = np.zeros((N, N)); bright_map = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        ys = slice(i * H // N, (i + 1) * H // N); xs = slice(j * W // N, (j + 1) * W // N)
        fp_map[i, j] = 100 * np.mean(p[ys, xs] & ~r[ys, xs])
        fn_map[i, j] = 100 * np.mean(~p[ys, xs] & r[ys, xs])
        bright_map[i, j] = bg[ys, xs].mean()

fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
axes[0].imshow(error_overlay(best_global_mask, ref_mask)); axes[0].set_title("Global errors\n(red=unwanted, blue=missing)")
axes[0].axis("off")
for k in range(1, N):
    for ax in (axes[0],):
        ax.axhline(k * H / N, color="green", lw=0.8); ax.axvline(k * W / N, color="green", lw=0.8)
for ax, mp, ttl, cm in [(axes[1], bright_map, "Paper brightness per cell", "viridis"),
                        (axes[2], fp_map, "Unwanted text % (false +)", "Reds"),
                        (axes[3], fn_map, "Missing text % (false −)", "Blues")]:
    im = ax.imshow(mp, cmap=cm); ax.set_title(ttl, fontsize=10)
    ax.set_xticks(range(N)); ax.set_xticklabels(cols_n); ax.set_yticks(range(N)); ax.set_yticklabels(rows_n)
    for i in range(N):
        for j in range(N):
            ax.text(j, i, f"{mp[i,j]:.1f}", ha="center", va="center", color="white" if mp[i, j] > mp.mean() else "black", fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

fi = np.unravel_index(fp_map.argmax(), fp_map.shape); ni = np.unravel_index(fn_map.argmax(), fn_map.shape)
dark_i  = np.unravel_index(bright_map.argmin(), bright_map.shape); bright_i = np.unravel_index(bright_map.argmax(), bright_map.shape)
print(f"Darkest paper region   : {rows_n[dark_i[0]]}-{cols_n[dark_i[1]]}   (mean background {bright_map[dark_i]:.0f})")
print(f"Brightest paper region : {rows_n[bright_i[0]]}-{cols_n[bright_i[1]]}   (mean background {bright_map[bright_i]:.0f})")
print(f"Most UNWANTED foreground (global T={best_global_T}): {rows_n[fi[0]]}-{cols_n[fi[1]]}  ({fp_map[fi]:.1f}% of that cell)")
print(f"Most MISSING text        (global T={best_global_T}): {rows_n[ni[0]]}-{cols_n[ni[1]]}  ({fn_map[ni]:.1f}% of that cell)")

**Reading the maps:** global thresholding fails at the two *extremes* of illumination:
- in the **darkest** part of the page, paper falls below `T` → large unwanted black regions (red);
- in the **brightest** part, faint/thin strokes stay above `T` → broken or missing characters (blue).

## Step 7 — Select the parameter combination that gives the most usable document mask

A small grid search over **global `T`** (every 5 grey levels) and **adaptive Gaussian `blockSize` / `C`** is scored against the proxy reference (F1), and the winners are inspected visually.

In [ ]:
# --- global sweep
sweep_T = list(range(40, 221, 5))
sweep_global = [(t, scores(cv2.threshold(gray, t, 255, cv2.THRESH_BINARY_INV)[1], ref_mask)["f1"]) for t in sweep_T]
best_T_sweep, best_T_f1 = max(sweep_global, key=lambda x: x[1])

# --- adaptive sweep (odd block sizes, several C)
blocks = [7, 11, 15, 21, 31, 41]
Cs = [2, 5, 8, 10, 15]
f1_grid = np.zeros((len(blocks), len(Cs)))
for i, b in enumerate(blocks):
    for j, c in enumerate(Cs):
        m = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, b, c)
        f1_grid[i, j] = scores(m, ref_mask)["f1"]
bi, bj = np.unravel_index(f1_grid.argmax(), f1_grid.shape)
BEST_BLOCK, BEST_C = blocks[bi], Cs[bj]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].plot([t for t, _ in sweep_global], [f for _, f in sweep_global], marker="o", ms=3)
axes[0].axvline(best_T_sweep, color="r", ls="--"); axes[0].set_title(f"Global: F1 vs T  (best T = {best_T_sweep}, F1 = {best_T_f1:.3f})")
axes[0].set_xlabel("global threshold T"); axes[0].set_ylabel("F1 vs proxy reference")
im = axes[1].imshow(f1_grid, cmap="viridis", aspect="auto")
axes[1].set_xticks(range(len(Cs))); axes[1].set_xticklabels(Cs); axes[1].set_yticks(range(len(blocks))); axes[1].set_yticklabels(blocks)
axes[1].set_xlabel("C"); axes[1].set_ylabel("blockSize"); axes[1].set_title(f"Adaptive Gaussian F1  (best: block={BEST_BLOCK}, C={BEST_C}, F1={f1_grid.max():.3f})")
for i in range(len(blocks)):
    for j in range(len(Cs)):
        axes[1].text(j, i, f"{f1_grid[i,j]:.2f}", ha="center", va="center", color="white", fontsize=8)
plt.colorbar(im, ax=axes[1]); plt.tight_layout(); plt.show()

adaptive_best = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, BEST_BLOCK, BEST_C)
print(f"Best global   : T = {best_T_sweep:>3}            F1 = {best_T_f1:.3f}")
print(f"Best adaptive : blockSize = {BEST_BLOCK}, C = {BEST_C}  F1 = {f1_grid.max():.3f}")

> ⚠️ The F1 score is measured against a *proxy* mask, so use it as guidance. **Always confirm the winner visually** in the comparison figure below – if a different neighbourhood/C looks cleaner to you, change `BEST_BLOCK`/`BEST_C` and re-run the following cells.

## Required output — Original → Global T1 → Global T2 → Global T3 → Adaptive

In [ ]:
QUESTION = "Why does a single threshold struggle when illumination changes across the image?"
ANSWER = ("A global threshold compares every pixel with the same intensity value, but a pixel's brightness is "
          "(paper brightness x ink absorption). When the lighting varies, dark text in the bright side can be "
          "brighter than blank paper in the dark side, so the two classes overlap in the histogram and no single T "
          "separates them. A low T loses text in dim areas, a high T turns dim paper into black blobs. Adaptive "
          "thresholding fixes this by using a local threshold from each pixel's neighbourhood, i.e. it compares "
          "text with the paper around it.")

fig, axes = plt.subplots(1, 5, figsize=(22, 5.6))
show(axes[0], gray, "Original")
for ax, (name, m) in zip(axes[1:4], global_masks.items()):
    show(ax, m, f"Global {name}  (T = {GLOBAL_T[name]})")
show(axes[4], adaptive_best, f"Adaptive Gaussian\n(block = {BEST_BLOCK}, C = {BEST_C})")

plt.subplots_adjust(bottom=0.30, top=0.90, wspace=0.05)
fig.text(0.5, 0.235, "Q: " + QUESTION, ha="center", va="top", fontsize=12, fontweight="bold")
fig.text(0.5, 0.17, textwrap.fill("A: " + ANSWER, 170), ha="center", va="top", fontsize=10)
fig.savefig(os.path.join(OUTPUT_DIR, "task1_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## Parameters selected and justification

In [ ]:
# --- gather final numbers
rows = []
for name, t in GLOBAL_T.items():
    s = global_scores[name]
    rows.append((f"Global {name}", f"T = {t}", s["text_%"], s["precision"], s["recall"], s["f1"]))
s_best_adapt = scores(adaptive_best, ref_mask)
rows.append(("Adaptive Gaussian", f"block = {BEST_BLOCK}, C = {BEST_C}", s_best_adapt["text_%"],
             s_best_adapt["precision"], s_best_adapt["recall"], s_best_adapt["f1"]))

print(f"{'Method':<20}{'Parameters':<22}{'text %':>8}{'prec.':>8}{'recall':>8}{'F1':>8}")
print("-" * 74)
for m, prm, tx, pr, rc, f1 in rows:
    print(f"{m:<20}{prm:<22}{tx:>8.1f}{pr:>8.3f}{rc:>8.3f}{f1:>8.3f}")

gain = s_best_adapt["f1"] - s_g["f1"]
print(f"\nAdaptive F1 {s_best_adapt['f1']:.3f} vs best of the three global thresholds (T={best_global_T}) {s_g['f1']:.3f}  ->  {gain:+.3f}")

# --- save final outputs
final_mask = adaptive_best
cv2.imwrite(os.path.join(OUTPUT_DIR, "task1_final_mask.png"), final_mask)
for name, m in global_masks.items():
    cv2.imwrite(os.path.join(OUTPUT_DIR, f"task1_global_{name}.png"), m)
with open(os.path.join(OUTPUT_DIR, "task1_parameters.json"), "w") as f:
    json.dump({"global_thresholds": GLOBAL_T, "best_global_of_three": {"name": best_global_name, "T": best_global_T},
               "adaptive": {"method": "ADAPTIVE_THRESH_GAUSSIAN_C", "blockSize": int(BEST_BLOCK), "C": int(BEST_C),
                            "type": "THRESH_BINARY_INV"}}, f, indent=2)
print("\nSaved to ./" + OUTPUT_DIR + ":", sorted(f for f in os.listdir(OUTPUT_DIR) if f.startswith("task1")))

**Justification**

| Parameter | Choice | Why |
|---|---|---|
| Grayscale loading | `IMREAD_GRAYSCALE` | Text/paper contrast is an intensity property; colour is not needed. |
| Threshold type | `THRESH_BINARY_INV` | Text is *darker* than paper; INV makes text white (255) so the mask is directly "text = foreground". |
| Global T1 / T2 / T3 | 90 / 127 / 160 | Low, mid and high values spread across the histogram to show the under-segmentation → over-segmentation trade-off. |
| Adaptive method | Gaussian-weighted mean | Pixels near the centre of the neighbourhood count more, which gives smoother thresholds than a plain mean. |
| `blockSize` | selected by sweep (`BEST_BLOCK`) | Must be large enough to span a full character stroke **plus** some paper, but small enough to follow the lighting change. |
| `C` | selected by sweep (`BEST_C`) | A positive `C` lowers the local threshold slightly so paper noise/texture is not marked as ink. |

**Conclusion for Task 1.** Every global threshold fails in at least one part of the page – low `T` loses text in the dim region, high `T`
turns the dim paper into unwanted black areas – because the paper brightness itself varies across the image. Adaptive thresholding keeps
the text in both the dark and bright regions because each pixel is judged against its **own neighbourhood**, so it is the most usable
document mask here.